# MarketSentiment — FinBERT fine-tune (Colab)

Fine-tunes FinBERT for **bullish / bearish / neutral** market sentiment with a
**class-weighted loss** to handle imbalance. Mirrors `train_finbert.py` in the repo.

Runtime: **Runtime -> Change runtime type -> GPU** (a free T4 is plenty; FinBERT is ~110M params).


## 1. Install deps

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

## 2. Load a public labeled dataset

`zeroshot/twitter-financial-news-sentiment` — finance tweets labeled bearish/bullish/neutral.
No scraping needed. (Swap in your own StockTwits-harvested CSV later.)

In [ ]:
import pandas as pd
from datasets import load_dataset

DATASET = 'zeroshot/twitter-financial-news-sentiment'
LABELMAP = {0: 'bearish', 1: 'bullish', 2: 'neutral'}

ds = load_dataset(DATASET)
frames = [ds[s].to_pandas()[['text', 'label']] for s in ds]
df = pd.concat(frames, ignore_index=True)
df['label'] = df['label'].map(LABELMAP)
df = df.dropna(subset=['text', 'label'])
print(len(df), 'rows')
print(df['label'].value_counts())

## 3. Encode, split, compute class weights

Inverse-frequency weights are the imbalance fix — the minority class stops being drowned out.

In [ ]:
import numpy as np, torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

labels = sorted(df['label'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
df['y'] = df['label'].map(label2id)
n_labels = len(labels)

train_df, eval_df = train_test_split(df, test_size=0.2, stratify=df['y'], random_state=42)
class_weights = compute_class_weight('balanced', classes=np.arange(n_labels), y=train_df['y'].to_numpy())
weight_tensor = torch.tensor(class_weights, dtype=torch.float)
print('labels:', label2id)
print('weights:', dict(zip(labels, class_weights.round(3))))

## 4. Tokenize + model

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

BASE = 'ProsusAI/finbert'
tok = AutoTokenizer.from_pretrained(BASE)

def to_ds(frame):
    d = Dataset.from_dict({'text': frame['text'].tolist(), 'labels': frame['y'].tolist()})
    return d.map(lambda b: tok(b['text'], truncation=True, max_length=128), batched=True,
                 remove_columns=['text'])

train_ds, eval_ds = to_ds(train_df), to_ds(eval_df)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=n_labels, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

## 5. Weighted-loss Trainer + train

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import precision_recall_fscore_support, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        y = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.CrossEntropyLoss(weight=weight_tensor.to(out.logits.device))(
            out.logits.view(-1, n_labels), y.view(-1))
        return (loss, out) if return_outputs else loss

def metrics(p):
    y_pred = np.argmax(p.predictions, axis=-1)
    pr, rc, f1, _ = precision_recall_fscore_support(p.label_ids, y_pred,
        labels=np.arange(n_labels), zero_division=0)
    m = {'macro_f1': float(f1.mean()), 'accuracy': float((y_pred == p.label_ids).mean())}
    for i, l in enumerate(labels): m[f'recall_{l}'] = float(rc[i])
    return m

args = TrainingArguments(output_dir='finbert-st', num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1',
    logging_steps=50, report_to='none')

# tokenizer not passed to Trainer (arg renamed to processing_class in transformers 5);
# the explicit data_collator handles padding, and we save the tokenizer separately.
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

## 6. Evaluate + save

**Minority-class recall** is the headline number — it proves the class weighting worked.

In [ ]:
preds = trainer.predict(eval_ds)
y_pred = np.argmax(preds.predictions, axis=-1)
print(classification_report(preds.label_ids, y_pred, target_names=labels, digits=3))

trainer.save_model('finbert-st'); tok.save_pretrained('finbert-st')

## 7. (Optional) Save to Google Drive

Then download `finbert-st/` and point the app at it: `MS_FINBERT_MODEL=path/to/finbert-st`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r finbert-st /content/drive/MyDrive/finbert-st
print('saved to Drive: MyDrive/finbert-st')